In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="08-ad-click-prediction",
    notebook_name="03_deep_ctr_models.ipynb"
)

# 03 -- Deep CTR Models: From Logistic Regression to DeepFM

**Goal:** Build and compare the key models used in ad click prediction, with actual PyTorch code.

---

## The Music Band Analogy

Think of each model as a music band:

- **Logistic Regression** = a solo guitarist. Talented, but can only play one note at a time.
- **Feature Cross + LR** = the guitarist learns a few chord combinations. Better, but limited by what chords the guitarist knows.
- **Factorization Machines (FM)** = the guitarist gets a loop pedal and can layer ALL pairs of notes together automatically.
- **Deep Neural Network (DNN)** = a full band (guitar + drums + bass + keyboard). Can play complex arrangements but sometimes misses subtle harmonies.
- **DeepFM** = the band PLUS the loop pedal. Best of both worlds -- captures both simple harmonies (FM) and complex arrangements (DNN).
- **DCN** = a band with a special "harmony detector" (cross network) alongside the standard musicians.
- **DIN** = a band that plays different songs for different audience members (attention mechanism).

Let's build them all.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# SYNTHETIC DATASET
# Simulates the feature space of an ad click prediction system
# ============================================================

np.random.seed(42)
torch.manual_seed(42)

N = 50000  # Training samples
N_SPARSE = 5  # Number of sparse categorical features
N_DENSE = 5   # Number of dense numerical features

# Sparse features: category IDs (small cardinalities for demo)
sparse_cardinalities = [100, 50, 20, 10, 7]  # user_id, ad_id, category, device, day
sparse_names = ['user_bucket', 'ad_bucket', 'category', 'device', 'day_of_week']

sparse_features = np.column_stack([
    np.random.randint(0, card, N) for card in sparse_cardinalities
])

# Dense features: numerical values
dense_names = ['user_ctr', 'ad_ctr', 'session_length', 'hour', 'position']
dense_features = np.column_stack([
    np.random.beta(2, 50, N),          # user_ctr
    np.random.beta(2, 50, N),          # ad_ctr
    np.random.exponential(5, N),        # session_length (minutes)
    np.random.randint(0, 24, N) / 24.0, # hour (normalized)
    np.random.randint(1, 6, N) / 5.0,  # position (normalized)
])

# Create labels with realistic non-linear interactions
# The "true model" has both pairwise and higher-order interactions
logits = (
    -3.0
    + 8.0 * dense_features[:, 0]           # user_ctr is king
    + 5.0 * dense_features[:, 1]           # ad_ctr matters a lot
    - 1.5 * dense_features[:, 4]           # position bias
    + 3.0 * dense_features[:, 0] * dense_features[:, 1]  # INTERACTION: user*ad CTR
    + np.where(sparse_features[:, 2] < 5, 0.5, -0.3)  # category effect
    + np.where((sparse_features[:, 3] < 3) & (dense_features[:, 3] > 0.7), 0.8, 0.0)  # device*time
    + np.random.normal(0, 0.3, N)
)
labels = (np.random.random(N) < 1 / (1 + np.exp(-logits))).astype(np.float32)

print(f"SYNTHETIC AD CLICK DATASET")
print(f"=" * 50)
print(f"Samples: {N:,}")
print(f"Sparse features: {N_SPARSE} ({sparse_names})")
print(f"Dense features:  {N_DENSE} ({dense_names})")
print(f"CTR: {labels.mean():.2%}")
print(f"\nSparse cardinalities: {dict(zip(sparse_names, sparse_cardinalities))}")

# Train/test split
idx = np.arange(N)
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42)

X_sparse_train = sparse_features[train_idx]
X_sparse_test = sparse_features[test_idx]
X_dense_train = dense_features[train_idx]
X_dense_test = dense_features[test_idx]
y_train = labels[train_idx]
y_test = labels[test_idx]

## Model 1: Logistic Regression Baseline

The simplest possible model. It computes a weighted sum of features and passes it through a sigmoid.

$$P(\text{click}) = \sigma(w_0 + \sum_i w_i x_i)$$

**Pros:** Fast, interpretable, great baseline.
**Cons:** Cannot learn feature interactions. If "young + gaming = high CTR" but neither alone predicts clicks, LR is stuck.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Encode sparse features as one-hot for sklearn LR
ohe = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
X_sparse_ohe_train = ohe.fit_transform(X_sparse_train)
X_sparse_ohe_test = ohe.transform(X_sparse_test)

from scipy.sparse import hstack
from scipy.sparse import csr_matrix

X_train_full = hstack([X_sparse_ohe_train, csr_matrix(X_dense_train)])
X_test_full = hstack([X_sparse_ohe_test, csr_matrix(X_dense_test)])

# Train LR
lr_model = LogisticRegression(max_iter=1000, C=1.0)
lr_model.fit(X_train_full, y_train)
lr_preds = lr_model.predict_proba(X_test_full)[:, 1]
lr_auc = roc_auc_score(y_test, lr_preds)
lr_logloss = log_loss(y_test, lr_preds)

print(f"LOGISTIC REGRESSION BASELINE")
print(f"=" * 40)
print(f"AUC-ROC:  {lr_auc:.4f}")
print(f"Log Loss: {lr_logloss:.4f}")
print(f"Features: {X_train_full.shape[1]} (sparse one-hot + dense)")
print(f"\nLR can only capture LINEAR relationships.")
print(f"It misses the user_ctr * ad_ctr interaction entirely.")

## Model 2: DeepFM (Factorization Machines + DNN)

DeepFM is the workhorse of modern ad click prediction. It combines:

1. **FM Component** -- captures ALL pairwise feature interactions efficiently
2. **DNN Component** -- captures higher-order, non-linear interactions

Both components share the same embedding layer, which is key to its efficiency.

```
                    ┌─────────────────┐
Sparse Features ──>│  Embedding Layer │──┬──> FM Component ──┐
                    │  (shared)        │  │   (pairwise)      │
Dense Features ───>│                  │  │                    ├──> Sigmoid ──> P(click)
                    └─────────────────┘  │                    │
                                         └──> DNN Component ─┘
                                              (higher-order)
```

In [ ]:
class FMLayer(nn.Module):
    """
    Factorization Machine interaction layer.
    
    12-year-old version:
    Imagine each feature gets a "personality vector" (embedding).
    The FM checks how well every pair of personalities get along
    (dot product). If user_42's personality and ad_travel's personality
    are similar, FM predicts a click.
    
    Staff-level detail:
    FM computes: sum_{i<j} <v_i, v_j> * x_i * x_j
    
    Using the efficient formula:
    0.5 * [||sum(v_i * x_i)||^2 - sum(||v_i * x_i||^2)]
    
    This reduces complexity from O(n^2 * k) to O(n * k).
    """
    def __init__(self):
        super().__init__()
    
    def forward(self, embeddings_list):
        """
        Args:
            embeddings_list: list of tensors, each [batch_size, embed_dim]
        Returns:
            FM interaction output [batch_size, 1]
        """
        # Stack embeddings: [batch_size, n_fields, embed_dim]
        stacked = torch.stack(embeddings_list, dim=1)
        
        # Efficient FM formula:
        # 0.5 * (sum_square - square_sum)
        sum_of_emb = stacked.sum(dim=1)        # [batch, embed_dim]
        sum_square = sum_of_emb ** 2            # [batch, embed_dim]
        square_sum = (stacked ** 2).sum(dim=1)  # [batch, embed_dim]
        
        # Sum over embedding dimensions, then take 0.5
        fm_out = 0.5 * (sum_square - square_sum).sum(dim=1, keepdim=True)
        return fm_out


class DeepFM(nn.Module):
    """
    DeepFM: Deep Factorization Machine for CTR Prediction.
    
    Combines FM (pairwise interactions) + DNN (higher-order interactions).
    Both share the same embedding layer.
    
    Staff-level interview points:
    - FM captures O(n^2) pairwise interactions efficiently
    - DNN captures arbitrary higher-order interactions
    - Shared embeddings reduce parameter count
    - End-to-end training (unlike GBDT+LR)
    - Supports continual learning via fine-tuning
    """
    
    def __init__(self, sparse_cardinalities, n_dense, embed_dim=8, 
                 hidden_dims=[128, 64, 32], dropout=0.2):
        super().__init__()
        
        self.n_sparse = len(sparse_cardinalities)
        self.n_dense = n_dense
        self.embed_dim = embed_dim
        
        # === Shared Embedding Layer ===
        # Each sparse feature gets its own embedding table
        self.embeddings = nn.ModuleList([
            nn.Embedding(card, embed_dim) for card in sparse_cardinalities
        ])
        
        # Project dense features to embedding space (so FM can process them)
        self.dense_to_embed = nn.Linear(1, embed_dim)  # Per-feature projection
        
        # === FM Component ===
        self.fm = FMLayer()
        
        # === Linear (first-order) Component ===
        # First-order weights for each sparse feature
        self.first_order_sparse = nn.ModuleList([
            nn.Embedding(card, 1) for card in sparse_cardinalities
        ])
        self.first_order_dense = nn.Linear(n_dense, 1)
        self.bias = nn.Parameter(torch.zeros(1))
        
        # === DNN Component ===
        dnn_input_dim = self.n_sparse * embed_dim + n_dense
        layers = []
        prev_dim = dnn_input_dim
        for hdim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hdim),
                nn.BatchNorm1d(hdim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = hdim
        layers.append(nn.Linear(prev_dim, 1))
        self.dnn = nn.Sequential(*layers)
    
    def forward(self, sparse_x, dense_x):
        """
        Args:
            sparse_x: [batch_size, n_sparse] LongTensor of category indices
            dense_x:  [batch_size, n_dense] FloatTensor of dense features
        Returns:
            P(click): [batch_size]
        """
        # --- Embedding lookup ---
        sparse_embeds = [emb(sparse_x[:, i]) for i, emb in enumerate(self.embeddings)]
        # sparse_embeds: list of [batch, embed_dim]
        
        # --- First-order term ---
        first_order = self.bias
        for i, fo_emb in enumerate(self.first_order_sparse):
            first_order = first_order + fo_emb(sparse_x[:, i]).squeeze(1)
        first_order = first_order + self.first_order_dense(dense_x)
        
        # --- FM term (second-order pairwise interactions) ---
        fm_out = self.fm(sparse_embeds)
        
        # --- DNN term (higher-order interactions) ---
        dnn_input = torch.cat(sparse_embeds + [dense_x], dim=1)
        dnn_out = self.dnn(dnn_input)
        
        # --- Combine all three components ---
        logit = first_order + fm_out + dnn_out
        return torch.sigmoid(logit).squeeze(1)


# Instantiate
deepfm = DeepFM(sparse_cardinalities, N_DENSE, embed_dim=8, hidden_dims=[64, 32])

total_params = sum(p.numel() for p in deepfm.parameters())
print(f"DeepFM Architecture")
print(f"=" * 50)
print(f"Sparse features: {N_SPARSE} (cardinalities: {sparse_cardinalities})")
print(f"Dense features:  {N_DENSE}")
print(f"Embedding dim:   8")
print(f"DNN layers:      [64, 32]")
print(f"Total parameters: {total_params:,}")
print(f"\nComponents:")
print(f"  First-order (LR): bias + sparse weights + dense weights")
print(f"  FM:               Pairwise interactions via embedding dot products")
print(f"  DNN:              Higher-order interactions via deep network")

In [ ]:
# === Training Loop ===

def train_model(model, X_sparse_train, X_dense_train, y_train,
                X_sparse_test, X_dense_test, y_test,
                epochs=20, batch_size=512, lr=0.001, model_name='Model'):
    """
    Train a CTR model and track metrics.
    """
    # Convert to tensors
    train_sparse = torch.LongTensor(X_sparse_train)
    train_dense = torch.FloatTensor(X_dense_train)
    train_labels = torch.FloatTensor(y_train)
    test_sparse = torch.LongTensor(X_sparse_test)
    test_dense = torch.FloatTensor(X_dense_test)
    
    train_dataset = TensorDataset(train_sparse, train_dense, train_labels)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    
    history = {'epoch': [], 'train_loss': [], 'test_auc': [], 'test_logloss': []}
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch_sparse, batch_dense, batch_labels in train_loader:
            optimizer.zero_grad()
            preds = model(batch_sparse, batch_dense)
            loss = criterion(preds, batch_labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # Evaluate
        model.eval()
        with torch.no_grad():
            test_preds = model(test_sparse, test_dense).numpy()
            test_preds = np.clip(test_preds, 1e-7, 1 - 1e-7)
            test_auc = roc_auc_score(y_test, test_preds)
            test_ll = log_loss(y_test, test_preds)
        
        avg_loss = total_loss / len(train_loader)
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(avg_loss)
        history['test_auc'].append(test_auc)
        history['test_logloss'].append(test_ll)
        
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | "
                  f"AUC: {test_auc:.4f} | LogLoss: {test_ll:.4f}")
    
    return history, test_preds


print(f"Training DeepFM...")
deepfm_model = DeepFM(sparse_cardinalities, N_DENSE, embed_dim=8, hidden_dims=[64, 32])
deepfm_history, deepfm_preds = train_model(
    deepfm_model, X_sparse_train, X_dense_train, y_train,
    X_sparse_test, X_dense_test, y_test,
    epochs=20, model_name='DeepFM'
)

## Model 3: DCN (Deep & Cross Network)

Google's approach (2017). Instead of using FM for pairwise interactions, DCN uses a **cross network** that explicitly computes feature crosses at each layer.

```
Cross Network (L layers):           Deep Network:
  x_{l+1} = x_0 * x_l^T * w_l      Standard DNN
            + b_l + x_l              (fully connected)
  
  Layer 0: original features
  Layer 1: 2nd-order crosses
  Layer 2: 3rd-order crosses
  ...
```

The cross network grows the interaction order by 1 at each layer, while keeping the same dimensionality.

In [ ]:
class CrossLayer(nn.Module):
    """
    One layer of the Cross Network.
    
    12-year-old version:
    Each cross layer takes the original features and the current
    features, multiplies them together in a special way, and adds
    the result back. It is like layering paint -- each layer adds
    one more level of complexity.
    """
    def __init__(self, input_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(input_dim, 1) * 0.01)
        self.bias = nn.Parameter(torch.zeros(input_dim))
    
    def forward(self, x0, xl):
        """x_{l+1} = x0 * (xl^T * w) + b + xl"""
        cross = x0 * torch.mm(xl, self.weight)  # [batch, dim]
        return cross + self.bias + xl


class DCN(nn.Module):
    """
    Deep & Cross Network for CTR Prediction.
    
    Staff-level interview points:
    - Cross network explicitly models feature interactions
    - L cross layers capture up to (L+1)-order interactions
    - Parameter-efficient: each cross layer adds only O(d) params
    - Parallel architecture: cross + deep are independent
    - Google proposed DCN V2 in 2020 (improved cross layer)
    """
    
    def __init__(self, sparse_cardinalities, n_dense, embed_dim=8,
                 n_cross_layers=3, hidden_dims=[64, 32], dropout=0.2):
        super().__init__()
        
        self.embeddings = nn.ModuleList([
            nn.Embedding(card, embed_dim) for card in sparse_cardinalities
        ])
        
        input_dim = len(sparse_cardinalities) * embed_dim + n_dense
        
        # Cross Network
        self.cross_layers = nn.ModuleList([
            CrossLayer(input_dim) for _ in range(n_cross_layers)
        ])
        
        # Deep Network
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.BatchNorm1d(h),
                          nn.ReLU(), nn.Dropout(dropout)])
            prev = h
        self.deep = nn.Sequential(*layers)
        
        # Final layer: concatenate cross + deep outputs
        self.output_layer = nn.Linear(input_dim + prev, 1)
    
    def forward(self, sparse_x, dense_x):
        # Embed sparse features
        embeds = [emb(sparse_x[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(embeds + [dense_x], dim=1)
        
        # Cross Network
        x0 = x
        xl = x
        for cross_layer in self.cross_layers:
            xl = cross_layer(x0, xl)
        
        # Deep Network
        deep_out = self.deep(x)
        
        # Combine
        combined = torch.cat([xl, deep_out], dim=1)
        logit = self.output_layer(combined)
        return torch.sigmoid(logit).squeeze(1)


print(f"Training DCN...")
dcn_model = DCN(sparse_cardinalities, N_DENSE, embed_dim=8, n_cross_layers=3, hidden_dims=[64, 32])
dcn_history, dcn_preds = train_model(
    dcn_model, X_sparse_train, X_dense_train, y_train,
    X_sparse_test, X_dense_test, y_test,
    epochs=20, model_name='DCN'
)

## Model 4: DIN (Deep Interest Network)

Alibaba's approach. The key insight: **a user's interest is not fixed -- it depends on the ad being shown.**

A user who has clicked on [travel, gaming, food] ads in the past has different relevance for a travel ad vs. a gaming ad. DIN uses an **attention mechanism** to weight the user's click history based on the target ad.

```
User clicked: [travel_ad, gaming_ad, food_ad, gaming_ad, travel_ad]
Target ad:    insurance_ad

Attention weights: [0.1, 0.05, 0.3, 0.05, 0.1]  -- food is most relevant to insurance?!
```

The attention-weighted sum gives a **target-aware user embedding**.

In [ ]:
class AttentionLayer(nn.Module):
    """
    Attention mechanism for DIN.
    
    12-year-old version:
    Imagine you are studying for a science test. You look through
    ALL your old notes, but you pay more attention to the notes
    about the topics on the test. DIN does the same thing -- it
    looks at ALL the user's past clicks, but focuses on the ones
    most relevant to the current ad.
    """
    def __init__(self, embed_dim, hidden_dim=32):
        super().__init__()
        # Input: target ad embedding, history item embedding, their element-wise product
        self.attention_net = nn.Sequential(
            nn.Linear(embed_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
    
    def forward(self, target_embed, history_embeds):
        """
        Args:
            target_embed: [batch, embed_dim] - the ad we want to score
            history_embeds: [batch, seq_len, embed_dim] - user's click history
        Returns:
            weighted_history: [batch, embed_dim] - attention-weighted user interest
        """
        seq_len = history_embeds.size(1)
        target_expanded = target_embed.unsqueeze(1).expand_as(history_embeds)
        
        # Compute attention input: [target, history, target * history]
        attn_input = torch.cat([
            target_expanded,
            history_embeds,
            target_expanded * history_embeds,
        ], dim=2)  # [batch, seq_len, embed_dim * 3]
        
        # Compute attention weights
        attn_weights = self.attention_net(attn_input).squeeze(2)  # [batch, seq_len]
        attn_weights = F.softmax(attn_weights, dim=1)
        
        # Weighted sum of history
        weighted = torch.bmm(attn_weights.unsqueeze(1), history_embeds).squeeze(1)
        return weighted, attn_weights


class DIN(nn.Module):
    """
    Deep Interest Network for CTR Prediction.
    
    Staff-level interview points:
    - Attention mechanism makes user representation target-aware
    - Unlike fixed user embeddings, DIN adapts to each candidate ad
    - Mini-batch aware regularization for sparse features
    - Can handle variable-length user histories
    - Used in production at Alibaba, handling 1B+ daily requests
    """
    
    def __init__(self, sparse_cardinalities, n_dense, embed_dim=8,
                 history_len=10, hidden_dims=[64, 32], dropout=0.2):
        super().__init__()
        
        self.embed_dim = embed_dim
        self.history_len = history_len
        
        # Embeddings (same as DeepFM)
        self.embeddings = nn.ModuleList([
            nn.Embedding(card, embed_dim) for card in sparse_cardinalities
        ])
        
        # Ad category embedding (shared for target and history)
        self.ad_embed = nn.Embedding(sparse_cardinalities[1], embed_dim)  # ad_bucket
        
        # Attention layer
        self.attention = AttentionLayer(embed_dim)
        
        # DNN
        input_dim = len(sparse_cardinalities) * embed_dim + n_dense + embed_dim  # + attention output
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.BatchNorm1d(h),
                          nn.ReLU(), nn.Dropout(dropout)])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.dnn = nn.Sequential(*layers)
    
    def forward(self, sparse_x, dense_x, history_x=None):
        batch_size = sparse_x.size(0)
        
        # Embed sparse features
        embeds = [emb(sparse_x[:, i]) for i, emb in enumerate(self.embeddings)]
        
        # Target ad embedding
        target_ad_embed = self.ad_embed(sparse_x[:, 1])  # ad_bucket
        
        # Simulate user click history (in production, this comes from logs)
        if history_x is None:
            history_x = torch.randint(0, self.ad_embed.num_embeddings,
                                     (batch_size, self.history_len))
        history_embeds = self.ad_embed(history_x)
        
        # Attention-weighted user interest
        user_interest, attn_weights = self.attention(target_ad_embed, history_embeds)
        
        # Concatenate everything
        dnn_input = torch.cat(embeds + [dense_x, user_interest], dim=1)
        logit = self.dnn(dnn_input)
        
        return torch.sigmoid(logit).squeeze(1)


print(f"Training DIN...")
din_model = DIN(sparse_cardinalities, N_DENSE, embed_dim=8, history_len=10, hidden_dims=[64, 32])
din_history, din_preds = train_model(
    din_model, X_sparse_train, X_dense_train, y_train,
    X_sparse_test, X_dense_test, y_test,
    epochs=20, model_name='DIN'
)

In [ ]:
# === GRAND COMPARISON ===

results = {
    'Logistic Regression': {
        'AUC': lr_auc, 'LogLoss': lr_logloss,
        'Params': X_train_full.shape[1] + 1,
    },
    'DeepFM': {
        'AUC': deepfm_history['test_auc'][-1],
        'LogLoss': deepfm_history['test_logloss'][-1],
        'Params': sum(p.numel() for p in deepfm_model.parameters()),
    },
    'DCN': {
        'AUC': dcn_history['test_auc'][-1],
        'LogLoss': dcn_history['test_logloss'][-1],
        'Params': sum(p.numel() for p in dcn_model.parameters()),
    },
    'DIN': {
        'AUC': din_history['test_auc'][-1],
        'LogLoss': din_history['test_logloss'][-1],
        'Params': sum(p.numel() for p in din_model.parameters()),
    },
}

print("\n" + "=" * 65)
print("GRAND MODEL COMPARISON")
print("=" * 65)
print(f"{'Model':<25} {'AUC':>8} {'LogLoss':>10} {'Params':>10}")
print("-" * 55)
for name, r in results.items():
    print(f"{name:<25} {r['AUC']:>8.4f} {r['LogLoss']:>10.4f} {r['Params']:>10,}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# AUC comparison
names = list(results.keys())
aucs = [r['AUC'] for r in results.values()]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
axes[0].bar(range(len(names)), aucs, color=colors)
axes[0].set_xticks(range(len(names)))
axes[0].set_xticklabels(names, rotation=30, ha='right', fontsize=9)
axes[0].set_ylabel('AUC-ROC', fontsize=12)
axes[0].set_title('Model Comparison: AUC', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(aucs):
    axes[0].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=9)

# Training curves
for name, hist, color in [('DeepFM', deepfm_history, '#4ECDC4'),
                           ('DCN', dcn_history, '#45B7D1'),
                           ('DIN', din_history, '#FFA07A')]:
    axes[1].plot(hist['epoch'], hist['test_auc'], '-o', color=color, label=name, markersize=3)
axes[1].axhline(lr_auc, color='#FF6B6B', linestyle='--', label='LR baseline')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Test AUC', fontsize=12)
axes[1].set_title('Training Progress', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Loss curves
for name, hist, color in [('DeepFM', deepfm_history, '#4ECDC4'),
                           ('DCN', dcn_history, '#45B7D1'),
                           ('DIN', din_history, '#FFA07A')]:
    axes[2].plot(hist['epoch'], hist['train_loss'], '-o', color=color, label=name, markersize=3)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Training Loss', fontsize=12)
axes[2].set_title('Training Loss', fontsize=12)
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Calibration of Predicted Probabilities

In ad systems, calibration is CRITICAL because:
- Revenue = pCTR x bid
- If pCTR = 0.10 but true probability = 0.01, advertisers are overcharged 10x
- The auction mechanism relies on ACCURATE probabilities, not just correct ranking

In [ ]:
from sklearn.calibration import calibration_curve

# === Calibration Analysis ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calibration curves for each model
for name, preds, color in [
    ('LR', lr_preds, '#FF6B6B'),
    ('DeepFM', deepfm_preds, '#4ECDC4'),
    ('DCN', dcn_preds, '#45B7D1'),
    ('DIN', din_preds, '#FFA07A'),
]:
    prob_true, prob_pred = calibration_curve(y_test, preds, n_bins=10, strategy='uniform')
    axes[0].plot(prob_pred, prob_true, 'o-', color=color, label=name, linewidth=2)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
axes[0].set_xlabel('Mean Predicted Probability', fontsize=12)
axes[0].set_ylabel('True Fraction of Positives', fontsize=12)
axes[0].set_title('Calibration Curves\n(Closer to diagonal = better calibrated)', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, max(0.3, max(lr_preds.max(), deepfm_preds.max()) + 0.05))
axes[0].set_ylim(0, max(0.3, max(lr_preds.max(), deepfm_preds.max()) + 0.05))

# Prediction distributions
for name, preds, color in [
    ('LR', lr_preds, '#FF6B6B'),
    ('DeepFM', deepfm_preds, '#4ECDC4'),
    ('DCN', dcn_preds, '#45B7D1'),
    ('DIN', din_preds, '#FFA07A'),
]:
    axes[1].hist(preds, bins=50, alpha=0.5, color=color, label=f'{name} (mean={preds.mean():.4f})')

axes[1].axvline(y_test.mean(), color='black', linestyle='--', linewidth=2,
               label=f'True CTR ({y_test.mean():.4f})')
axes[1].set_xlabel('Predicted P(click)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Prediction Distributions', fontsize=12)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("A well-calibrated model means: when it predicts 5%, about")
print("5% of those impressions actually get clicked.")
print("\nCalibration techniques:")
print("  1. Platt scaling: fit a sigmoid on validation predictions")
print("  2. Isotonic regression: non-parametric calibration")
print("  3. Temperature scaling: single parameter T, divide logits by T")

## Interview Cheat Sheet: Deep CTR Models

### The 30-Second Model Selection Answer

> "I would start with **Logistic Regression** as a baseline, then experiment with **DeepFM** and **DCN**. DeepFM combines factorization machines for efficient pairwise interactions with a DNN for higher-order features. DCN uses a cross network that explicitly learns feature crosses. Both are widely used in production at major tech companies. For scenarios with rich user behavior sequences, I would also consider **DIN** which uses attention to create target-aware user representations."

### Model Summary Table

| Model | Key Idea | Pairwise | Higher-order | Continual Learning | Best For |
|-------|----------|----------|-------------|-------------------|----------|
| LR | Linear combination | No | No | Yes | Baseline |
| FM | Embedding dot products | Yes (all) | No | Yes | Sparse features |
| DeepFM | FM + DNN | Yes (all) | Yes | Yes | General CTR |
| DCN | Cross network + DNN | Yes (explicit) | Yes | Yes | Feature crosses |
| DIN | Attention on history | Yes | Yes | Yes | Rich user history |

### Key Phrases to Drop

- "FM captures ALL pairwise interactions in O(nk) time using the **efficient sum-of-squares formula**."
- "DeepFM shares embeddings between FM and DNN, reducing parameters and enabling **end-to-end training**."
- "DCN's cross network grows the **interaction order by 1 per layer** while maintaining the same dimensionality."
- "DIN's attention mechanism creates a **target-aware user representation** that adapts to each candidate ad."
- "Unlike GBDT, all these models support **continual learning** via fine-tuning on new data."

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)